### 1. Corpus

**Corpus কী?**

Corpus হলো text data-র একটা large, structured collection — যেটা NLP model train বা analyze করার জন্য ব্যবহার হয়। একটা corpus-এ হাজার হাজার বা মিলিয়ন মিলিয়ন sentence/document থাকতে পারে।

**IMDB dataset কি corpus?**

হ্যাঁ, IMDB movie review dataset একটা corpus, কারণ এটা হাজার হাজার real movie review-এর collection, প্রতিটা review একটা text document, আর পুরো dataset মিলে sentiment analysis task-এর জন্য একটা domain-specific corpus তৈরি করে।

### 2. Lowercasing

**কেন lowercasing করি?**

যাতে PHITRON, Phitron, phitron — এই সব variation model-এর কাছে same word হিসেবে treat হয়। Case sensitivity vocabulary-তে duplicate entry তৈরি করে, model সেটাকে আলাদা word ভাবে।

**Vocabulary size কমে কীভাবে?**

`PHITRON` আর `phitron` — lowercasing ছাড়া এগুলো দুইটা আলাদা token/vocabulary entry হতো। Lowercase করলে দুইটাই একটা entry-তে merge হয়ে যায়, ফলে vocabulary size কমে এবং model efficiently generalize করতে পারে।

### 3. HTML Tag Removal



<p>Welcome to <b>PHITRON</b>!</p>

**Regex**

In [3]:
import re
text = "<p>Welcome to <b>PHITRON</b>!</p>"
clean_text = re.sub(r'<.*?>', '', text)
print(clean_text)
# "Welcome to PHITRON!"

Welcome to PHITRON!


`<.*?>` **কেন কাজ করে?**

+ `<` এবং `>` — HTML tag-এর opening আর closing bracket match করে
+ `.*?` — মাঝখানের যেকোনো character (tag name, attribute) match করে

`?` এর role (non-greedy matching): `?` ছাড়া (`<.*>`) regex greedy হয়ে যেত — মানে প্রথম `<` থেকে শুরু করে শেষ `>` পর্যন্ত সব একসাথে match করত, ফলে `<p>Welcome to <b>PHITRON</b>!</p>` পুরোটাই একটা match হয়ে যেত এবং পুরো text মুছে যেত। `?` দিয়ে lazy/non-greedy matching হয় — প্রতিটা `<...>` আলাদা আলাদাভাবে match করে, তাই ভেতরের text (`Welcome to, PHITRON`) ঠিক থাকে।

### 4. URL Removal

In [5]:
text = "For more details visit https://phitron.io or www.phitron.io"
clean_text = re.sub(r'https?://\S+|www\S+', '', text)
print(clean_text)

For more details visit  or 


### 5. Punctuation Removal

In [6]:
import string
text.translate(str.maketrans('', '', string.punctuation))

'For more details visit httpsphitronio or wwwphitronio'

**string.punctuation কী করে?**

এটা একটা built-in string যেখানে সব common punctuation character থাকে (`!"#$%&'()*+,-./:;<=>?@[\]^_{|}~`)।

`maketrans()`**-এর প্রথম দুইটা argument empty কেন?**

`maketrans(x, y, z)` — এখানে `x`-এর প্রতিটা character `y`-এর corresponding character দিয়ে replace হয়, আর `z`-এ থাকা প্রতিটা character delete হয়। যেহেতু আমরা কোনো character replace করতে চাই না, শুধু delete করতে চাই — তাই প্রথম দুইটা empty string দেওয়া হয়েছে।

**Punctuation না সরালে কী হয়?**

Tokenization-এর সময় `amazing!` আর `amazing` আলাদা token হয়ে যাবে, `Dhaka!`, `Dhaka`. আলাদা entry হবে vocabulary-তে — ফলে unnecessary vocabulary size বাড়বে এবং model একই word-কে ভিন্ন ভিন্ন token ভাবে treat করবে।

### 6. Stopword Removal

In [7]:
stopwords = "I am studying Machine Learning in the university."

**Stopwords কী?**

Stopwords হলো এমন common word (`like` `I`, `am`, `in`, `the`, `is`, `at`) যেগুলো খুব বেশি frequent কিন্তু sentence-এর meaning বা sentiment-এ তেমন contribute করে না।

কোন words remove হবে: `I`, `am`, `in`, `the`

থাকবে: `studying`, `Machine`, `Learning`, `university`

**Set কেন ব্যবহার করি, list না?**

In [10]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
print(list(stop_words)[:10])

['yourself', 'itself', 'has', "shan't", 'did', "i've", "they'll", 'needn', "needn't", "i'd"]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


List-এ membership check (`word in stop_words`) করতে O(n) time লাগে, কিন্তু set-এ hashing-এর কারণে O(1) time লাগে। যেহেতু stopword removal-এ প্রতিটা word-এর জন্য এই check বার বার হয়, set ব্যবহার করলে performance অনেক দ্রুত হয় — বিশেষত বড় corpus-এর ক্ষেত্রে।

### 7. Tokenization

**Tokenization কী?**

Text-কে ছোট ছোট unit (word, subword বা sentence)-এ ভাঙাকে tokenization বলে। এটা NLP pipeline-এর একটা fundamental step, কারণ model raw text না, token-based input নেয়।

In [13]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
text = "I am going to visit Dhaka!"
text.split()
word_tokenize(text)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['I', 'am', 'going', 'to', 'visit', 'Dhaka', '!']

`split()` শুধু whitespace ধরে ভাঙে — punctuation word-এর সাথে লেগে থাকে `(Dhaka!)`। `word_tokenize()` linguistic rule-based, তাই punctuation-কে আলাদা token হিসেবে split করে।

কোনটা punctuation ভালো handle করে? `word_tokenize()` — কারণ এটা punctuation আর word আলাদা করে দেয়, ফলে downstream processing (stopword removal, stemming) আরও clean হয়।

### 8. Stemming

**Stemming কী?**

Stemming হলো word-এর suffix কেটে root/base form বের করার crude, rule-based process। এটা linguistic meaning বোঝে না, শুধু pattern অনুযায়ী কাটে।

**Dictionary word না পাওয়ার কারণ:** Stemming rule-based cutting করে, meaning বা grammar বোঝে না। তাই `studies` → `studi` বা `happiness` → `happi`-এর মতো output আসে, যেগুলো actual English dictionary word না। এটাই stemming-এর একটা limitation।

### 9. Lemmatization

`running`, `children`, `better`, `studies`

**Lemmatization কী?**

Lemmatization হলো word-কে vocabulary আর morphological analysis (POS context সহ) ব্যবহার করে proper dictionary base form (lemma)-এ convert করা।

**কেন lemmatization বেশি accurate?**

Lemmatization vocabulary ও grammar rule ব্যবহার করে, তাই সবসময় valid dictionary word দেয় (`better` → `good`, `children` → `child`)। Stemming শুধু blind suffix-stripping করে, তাই ভুল বা non-dictionary word তৈরি হতে পারে (`studies` → `studi`)।

### 10. Complete NLP Pipeline

In [17]:
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

text = "OMG!!! I was studying NLP at PHITRON 😊. Visit https//phitron.io for more details."

# LowerCasing
text = text.lower()

# URL Removal
import re
text = re.sub(r'https?://\S+|www\.\S+', '', text)

# Emoji Removal
text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF]', '', text)

# punctuation removal
import string
text = text.translate(str.maketrans('', '', string.punctuation))

# Tokenization
from nltk.tokenize import word_tokenize
tokens = word_tokenize(text)

# Stopword Removal
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
tokens = [w for w in tokens if w not in stop_words]

# Stemming
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in tokens]

# Lemmatization
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w) for w in tokens]

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
